In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def load_and_preprocess_data():
    """Load and preprocess the dataset"""
    try:
        df = pd.read_csv("content/train.csv")
        print("Data loaded successfully. First 5 rows:")
        display(df.head(5))

        # Preprocessing
        X = df[['Time_spent_Alone', 'Stage_fear', 'Social_event_attendance',
                'Going_outside', 'Drained_after_socializing',
                'Friends_circle_size', 'Post_frequency']]
        y = df['Personality']

        # Fill missing values
        fill_values = {
            'Time_spent_Alone': X['Time_spent_Alone'].mean(),
            'Stage_fear': X['Stage_fear'].mode()[0],
            'Social_event_attendance': X['Social_event_attendance'].mean(),
            'Going_outside': X['Going_outside'].mean(),
            'Drained_after_socializing': X['Drained_after_socializing'].mode()[0],
            'Friends_circle_size': X['Friends_circle_size'].mean(),
            'Post_frequency': X['Post_frequency'].mean()
        }

        X = X.fillna(fill_values)

        return X, y, df

    except Exception as e:
        print(f"Error loading or processing data: {e}")
        return None, None, None

def train_model(X, y):
    """Train the classification model"""
    try:
        # Define feature types
        numerical = ['Time_spent_Alone', 'Social_event_attendance',
                    'Going_outside', 'Friends_circle_size', 'Post_frequency']
        categorical = ['Stage_fear', 'Drained_after_socializing']

        # Preprocessing pipeline
        preprocessor = ColumnTransformer([
            ('num', StandardScaler(), numerical),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
        ])

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y)

        # Create and train model
        model = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                class_weight='balanced'
            ))
        ])

        model.fit(X_train, y_train)

        # Evaluate model
        print("\nModel Evaluation on Test Set:")
        y_pred = model.predict(X_test)
        print(classification_report(y_test, y_pred))

        # Plot confusion matrix
        plt.figure(figsize=(8, 6))
        sns.heatmap(confusion_matrix(y_test, y_pred),
                   annot=True, fmt='d', cmap='Blues')
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.show()

        return model

    except Exception as e:
        print(f"Error during model training: {e}")
        return None

def get_valid_input(prompt, input_type=float, min_val=None, max_val=None):
    """Get valid user input with type checking and range validation"""
    while True:
        try:
            value = input(prompt)
            if input_type == str:
                return value

            value = input_type(value)
            if min_val is not None and value < min_val:
                print(f"Value must be at least {min_val}")
                continue
            if max_val is not None and value > max_val:
                print(f"Value must be at most {max_val}")
                continue
            return value
        except ValueError:
            print(f"Please enter a valid {input_type.__name__}")

def predict_personality(model):
    """Get user input and make prediction"""
    try:
        print("\nEnter personality trait information:")

        inputs = {
            'Time_spent_Alone': get_valid_input(
                "Time spent alone (hours/week, 0-168): ",
                float, 0, 168),
            'Stage_fear': get_valid_input(
                "Stage fear (Yes/No): ", str).capitalize(),
            'Social_event_attendance': get_valid_input(
                "Social event attendance (times/month, 0-31): ",
                float, 0, 31),
            'Going_outside': get_valid_input(
                "Going outside (times/week, 0-7): ",
                float, 0, 7),
            'Drained_after_socializing': get_valid_input(
                "Drained after socializing (Yes/No): ", str).capitalize(),
            'Friends_circle_size': get_valid_input(
                "Friend circle size (number of close friends, 0-100): ",
                float, 0, 100),
            'Post_frequency': get_valid_input(
                "Post frequency (posts/week, 0-100): ",
                float, 0, 100)
        }

        # Create DataFrame
        new_input = pd.DataFrame([inputs])

        # Make prediction
        prediction = model.predict(new_input)

        print(f"\nPredicted Personality: {prediction[0]}")

    except Exception as e:
        print(f"Error during prediction: {e}")

def main():
    """Main function to run the program"""
    print("Personality Prediction System\n")

    # Load and preprocess data
    X, y, df = load_and_preprocess_data()
    if X is None:
        return

    # Train model
    model = train_model(X, y)
    if model is None:
        return

    # Make predictions
    while True:
        predict_personality(model)
        if input("\nMake another prediction? (y/n): ").lower() != 'y':
            break

if __name__ == "__main__":
    main()
